In [1]:
{
 "nbformat": 4,
 "nbformat_minor": 0,
 "metadata": {
  "colab": {
   "provenance": [],
   "gpuType": "T4"
  },
  "kernelspec": {
   "name": "python3",
   "display_name": "Python 3"
  },
  "language_info": {
   "name": "python"
  },
  "accelerator": "GPU"
 },
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# TerrariaGuide - 8B Fine-Tune\n",
    "Run each cell in order. Make sure **Runtime > Change runtime type** is set to **T4 GPU** before starting."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "# Cell 1: Install dependencies\n",
    "# Set memory allocator flag before any CUDA ops to reduce fragmentation OOMs\n",
    "import os\n",
    "os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'\n",
    "\n",
    "!pip install unsloth trl transformers datasets accelerate bitsandbytes peft sentencepiece -q"
   ],
   "execution_count": null,
   "outputs": []
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "# Cell 2: Mount Google Drive\n",
    "from google.colab import drive\n",
    "drive.mount('/content/drive')\n",
    "\n",
    "import os\n",
    "TRAINING_FILE = '/content/drive/MyDrive/TerrariaGuide/terraria_training_pairs.jsonl'\n",
    "OUTPUT_DIR = '/content/drive/MyDrive/TerrariaGuide/models/terraria-guide-8b'\n",
    "\n",
    "if os.path.exists(TRAINING_FILE):\n",
    "    print(f'Training file found!')\n",
    "    print(f'Lines: {sum(1 for _ in open(TRAINING_FILE))}')\n",
    "else:\n",
    "    print(f'ERROR: Training file not found at {TRAINING_FILE}')\n",
    "    print('Please upload terraria_training_pairs.jsonl to your Drive at TerrariaGuide/terraria_training_pairs.jsonl')"
   ],
   "execution_count": null,
   "outputs": []
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "# Cell 3: Check GPU\n",
    "import torch\n",
    "print(f'GPU: {torch.cuda.get_device_name(0)}')\n",
    "print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')\n",
    "print(f'CUDA: {torch.version.cuda}')"
   ],
   "execution_count": null,
   "outputs": []
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "# Cell 4: Load model\n",
    "# FIX: lora_alpha raised to 32 (2x r=16) for better convergence\n",
    "from unsloth import FastLanguageModel\n",
    "\n",
    "MAX_SEQ_LENGTH = 1024\n",
    "MODEL_NAME = 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit'\n",
    "\n",
    "print('Loading base model...')\n",
    "model, tokenizer = FastLanguageModel.from_pretrained(\n",
    "    model_name=MODEL_NAME,\n",
    "    max_seq_length=MAX_SEQ_LENGTH,\n",
    "    dtype=None,\n",
    "    load_in_4bit=True,\n",
    ")\n",
    "\n",
    "model = FastLanguageModel.get_peft_model(\n",
    "    model,\n",
    "    r=16,\n",
    "    target_modules=[\n",
    "        'q_proj', 'k_proj', 'v_proj', 'o_proj',\n",
    "        'gate_proj', 'up_proj', 'down_proj',\n",
    "    ],\n",
    "    lora_alpha=32,          # FIX: was 16, now 2x r for better convergence\n",
    "    lora_dropout=0,\n",
    "    bias='none',\n",
    "    use_gradient_checkpointing='unsloth',\n",
    "    random_state=42,\n",
    ")\n",
    "print('Model ready!')"
   ],
   "execution_count": null,
   "outputs": []
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "# Cell 5: Load and format dataset\n",
    "# FIX: Use Llama 3.1 instruct chat template instead of Alpaca-style prompts\n",
    "#      This matches the format the base model was trained on.\n",
    "# FIX: Filter by tokenized length instead of character count to properly\n",
    "#      respect MAX_SEQ_LENGTH=1024 tokens.\n",
    "from datasets import load_dataset\n",
    "\n",
    "def format_prompt(example):\n",
    "    # Use the tokenizer's built-in chat template for Llama 3.1 Instruct\n",
    "    messages = [\n",
    "        {\n",
    "            'role': 'system',\n",
    "            'content': 'You are TerrariaGuide, an expert assistant for the game Terraria. Answer questions accurately about items, crafting, bosses, enemies, biomes, and game mechanics.'\n",
    "        },\n",
    "        {'role': 'user',    'content': example['instruction']},\n",
    "        {'role': 'assistant', 'content': example['output']},\n",
    "    ]\n",
    "    return {\n",
    "        'text': tokenizer.apply_chat_template(\n",
    "            messages,\n",
    "            tokenize=False,\n",
    "            add_generation_prompt=False\n",
    "        )\n",
    "    }\n",
    "\n",
    "def is_within_seq_length(example):\n",
    "    # FIX: filter by actual token count, not character count\n",
    "    token_count = len(tokenizer(example['text'], add_special_tokens=False)['input_ids'])\n",
    "    return token_count <= MAX_SEQ_LENGTH\n",
    "\n",
    "print('Loading dataset...')\n",
    "dataset = load_dataset('json', data_files=TRAINING_FILE, split='train')\n",
    "dataset = dataset.map(format_prompt)\n",
    "before = len(dataset)\n",
    "dataset = dataset.filter(is_within_seq_length)\n",
    "print(f'Dataset size: {len(dataset)} pairs ({before - len(dataset)} filtered for exceeding {MAX_SEQ_LENGTH} tokens)')"
   ],
   "execution_count": null,
   "outputs": []
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "# Cell 6: Train\n",
    "# FIX: batch_size reduced 2->1 and gradient_accumulation_steps increased 4->8\n",
    "#      to maintain the same effective batch size (8) while halving peak VRAM usage.\n",
    "#      This resolves the OOM on T4 (15GB).\n",
    "# FIX: packing=True enabled — packs multiple short examples into each sequence\n",
    "#      window for significantly faster training throughput.\n",
    "from trl import SFTTrainer\n",
    "from transformers import TrainingArguments\n",
    "import os\n",
    "\n",
    "os.makedirs(OUTPUT_DIR, exist_ok=True)\n",
    "\n",
    "training_args = TrainingArguments(\n",
    "    output_dir=OUTPUT_DIR,\n",
    "    per_device_train_batch_size=1,      # FIX: was 2, reduced to avoid OOM on T4\n",
    "    gradient_accumulation_steps=8,      # FIX: was 4, increased to keep effective batch=8\n",
    "    warmup_steps=50,\n",
    "    num_train_epochs=1,\n",
    "    learning_rate=2e-4,\n",
    "    fp16=not torch.cuda.is_bf16_supported(),\n",
    "    bf16=torch.cuda.is_bf16_supported(),\n",
    "    logging_steps=25,\n",
    "    save_steps=500,\n",
    "    save_total_limit=2,\n",
    "    optim='adamw_8bit',\n",
    "    weight_decay=0.01,\n",
    "    lr_scheduler_type='cosine',\n",
    "    seed=42,\n",
    "    report_to='none',\n",
    ")\n",
    "\n",
    "trainer = SFTTrainer(\n",
    "    model=model,\n",
    "    tokenizer=tokenizer,\n",
    "    train_dataset=dataset,\n",
    "    dataset_text_field='text',\n",
    "    max_seq_length=MAX_SEQ_LENGTH,\n",
    "    dataset_num_proc=2,\n",
    "    packing=True,           # FIX: was False, packing speeds up training on short examples\n",
    "    args=training_args,\n",
    ")\n",
    "\n",
    "print('Starting training...')\n",
    "trainer.train()"
   ],
   "execution_count": null,
   "outputs": []
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "# Cell 7: Save model to Drive\n",
    "print('Saving model to Google Drive...')\n",
    "model.save_pretrained(OUTPUT_DIR)\n",
    "tokenizer.save_pretrained(OUTPUT_DIR)\n",
    "print(f'Model saved to {OUTPUT_DIR}')"
   ],
   "execution_count": null,
   "outputs": []
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "# Cell 8: Quick test\n",
    "# FIX: Inference now uses the same Llama 3.1 chat template as training\n",
    "FastLanguageModel.for_inference(model)\n",
    "\n",
    "def ask(question):\n",
    "    messages = [\n",
    "        {\n",
    "            'role': 'system',\n",
    "            'content': 'You are TerrariaGuide, an expert assistant for the game Terraria. Answer questions accurately about items, crafting, bosses, enemies, biomes, and game mechanics.'\n",
    "        },\n",
    "        {'role': 'user', 'content': question},\n",
    "    ]\n",
    "    # add_generation_prompt=True appends the assistant header so the model knows to respond\n",
    "    prompt = tokenizer.apply_chat_template(\n",
    "        messages,\n",
    "        tokenize=False,\n",
    "        add_generation_prompt=True\n",
    "    )\n",
    "    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')\n",
    "    with torch.no_grad():\n",
    "        outputs = model.generate(\n",
    "            **inputs,\n",
    "            max_new_tokens=512,\n",
    "            temperature=0.7,\n",
    "            top_p=0.9,\n",
    "            do_sample=True,\n",
    "            pad_token_id=tokenizer.eos_token_id,\n",
    "        )\n",
    "    # Decode only the newly generated tokens (skip the prompt)\n",
    "    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]\n",
    "    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()\n",
    "\n",
    "print('Test 1:', ask('What does the Eye of Cthulhu drop in Terraria?'))\n",
    "print()\n",
    "print('Test 2:', ask('How do I craft a Molten Pickaxe in Terraria?'))\n",
    "print()\n",
    "print('Test 3:', ask('What is the Wall of Flesh in Terraria?'))"
   ],
   "execution_count": null,
   "outputs": []
  }
 ]
}

NameError: name 'null' is not defined